<a href="https://colab.research.google.com/github/whistle-hikhi/Logistic-Regression/blob/main/emotion_tone_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!nvidia-smi

Wed Apr  2 05:33:38 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
%load_ext cuml.accel

[2025-04-02 05:33:58.781] [CUML] [info] cuML: Installed accelerator for sklearn.
[2025-04-02 05:34:41.741] [CUML] [info] cuML: Installed accelerator for umap.
[2025-04-02 05:34:41.844] [CUML] [info] cuML: Installed accelerator for hdbscan.
[2025-04-02 05:34:41.844] [CUML] [info] cuML: Successfully initialized accelerator.


In [51]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import nltk
nltk.download('punkt')

# Step 1: Create a small custom dataset
data = {
    "text": [
        "I’m so happy today!", "This is the worst day ever.", "I’m furious about this!",
        "It’s just another day.", "Wow, I’m so excited!", "I love this so much!",
        "Why does this always happen?", "Feeling calm and okay.", "This makes me so mad!",
        "Yay, great news!"
    ],
    "emotion": [
        "Happy", "Sad", "Angry", "Neutral", "Excited",
        "Happy", "Sad", "Neutral", "Angry", "Excited"
    ]
}
df = pd.DataFrame(data)

# Step 2: Preprocess the text
vectorizer = TfidfVectorizer(max_features=100, stop_words='english')
X = vectorizer.fit_transform(df['text']).toarray()
y = df['emotion'].values  # Convert to NumPy array

# Ensure X is float32 (compatible with cuML if used)
X = X.astype(np.float32)

# Convert y to categorical codes (numeric labels) for compatibility
emotion_labels = np.unique(y)  # Unique emotion names
y_numeric = pd.factorize(y)[0]  # Convert to 0, 1, 2, ... (int64)

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_numeric, test_size=0.2, random_state=42)

# Step 3: Train the softmax regression model
print("Training the model...")
model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=200)
model.fit(X_train, y_train)

# Step 4: Test the model
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model accuracy: {accuracy * 100:.2f}%")

# Step 5: Predict on new text
def predict_emotion(text):
    text_vector = vectorizer.transform([text]).toarray().astype(np.float32)
    prediction_numeric = model.predict(text_vector)[0]
    return emotion_labels[prediction_numeric]  # Map back to emotion name

# Test it out
new_texts = [
    "I can’t believe how awesome this is!",
    "This is so annoying.",
    "Nothing special today."
]
for text in new_texts:
    emotion = predict_emotion(text)
    print(f"Text: '{text}' -> Predicted Emotion: {emotion}")

Training the model...
Model accuracy: 0.00%
Text: 'I can’t believe how awesome this is!' -> Predicted Emotion: Neutral
Text: 'This is so annoying.' -> Predicted Emotion: Neutral
Text: 'Nothing special today.' -> Predicted Emotion: Angry


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
